In [1]:
import polars as pl

from Python import statcast

In [2]:
years = [2025]

In [3]:
print(statcast.STRIKEOUT_EVENTS)
print(statcast.NON_PA_EVENTS)
print(statcast.WHIFF_DESCRIPTIONS)
print(statcast.SWING_DESCRIPTIONS, statcast.CONTACT_DESCRIPTIONS)

frozenset({'strikeout', 'strikeout_double_play'})
frozenset({'stolen_base_3b', 'caught_stealing_home', 'pickoff_2b', 'stolen_base_2b', 'pickoff_3b', 'pickoff_caught_stealing_2b', 'other_out', 'pickoff_1b', 'wild_pitch', 'runner_double_play', 'pickoff_caught_stealing_home', 'caught_stealing_3b', 'caught_stealing_2b', 'stolen_base_home', 'passed_ball', 'pickoff_caught_stealing_3b'})
frozenset({'foul_tip', 'swinging_strike', 'swinging_strike_blocked'})
frozenset({'foul_tip', 'swinging_strike', 'hit_into_play', 'foul', 'swinging_strike_blocked'}) frozenset({'hit_into_play', 'foul'})


In [4]:
raw = statcast.load_statcast_years(years, columns=None)
raw.shape, raw.columns

((712528, 119),
 ['pitch_type',
  'game_date',
  'release_speed',
  'release_pos_x',
  'release_pos_z',
  'player_name',
  'batter',
  'pitcher',
  'events',
  'description',
  'spin_dir',
  'spin_rate_deprecated',
  'break_angle_deprecated',
  'break_length_deprecated',
  'zone',
  'des',
  'game_type',
  'stand',
  'p_throws',
  'home_team',
  'away_team',
  'type',
  'hit_location',
  'bb_type',
  'balls',
  'strikes',
  'game_year',
  'pfx_x',
  'pfx_z',
  'plate_x',
  'plate_z',
  'on_3b',
  'on_2b',
  'on_1b',
  'outs_when_up',
  'inning',
  'inning_topbot',
  'hc_x',
  'hc_y',
  'tfs_deprecated',
  'tfs_zulu_deprecated',
  'umpire',
  'sv_id',
  'vx0',
  'vy0',
  'vz0',
  'ax',
  'ay',
  'az',
  'sz_top',
  'sz_bot',
  'hit_distance_sc',
  'launch_speed',
  'launch_angle',
  'effective_speed',
  'release_spin_rate',
  'release_extension',
  'game_pk',
  'fielder_2',
  'fielder_3',
  'fielder_4',
  'fielder_5',
  'fielder_6',
  'fielder_7',
  'fielder_8',
  'fielder_9',
  'releas

In [5]:
pa = statcast.plate_appearances(raw)
pa.shape
pa.select(["game_pk","at_bat_number"]).is_duplicated().sum()  # should be 0

0

In [6]:
flagged = statcast.add_event_flags(raw)
flagged.select(["is_pa","is_k","is_bb","is_hbp","is_hr","is_hit","is_whiff","is_called_strike"]).sum()

is_pa,is_k,is_bb,is_hbp,is_hr,is_hit,is_whiff,is_called_strike
u32,u32,u32,u32,u32,u32,u32,u32
183245,40645,15379,1928,5650,40138,85419,115574


In [7]:
disc = statcast.add_plate_discipline_flags(statcast.add_event_flags(raw))

counts = disc.group_by("pitcher").agg(
    pl.len().alias("Pitches"),
    pl.col("is_whiff").sum().alias("Whiffs"),
    *statcast.discipline_count_exprs(),
)
counts.head()

pitcher,Pitches,Whiffs,InZone,OutZone,Swings,Chases,ZSwings,Contacts,ZContacts,OContacts
i64,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
642585,629,101,289,338,282,84,198,181,144,37
700249,1976,213,996,975,881,259,622,668,522,146
681751,119,24,64,55,64,17,47,40,37,3
622780,72,6,33,39,30,7,23,24,19,5
642180,18,0,5,13,5,2,3,5,3,2


In [8]:
woba_check = pa.group_by("batter").agg(
    pl.len().alias("PA"),
    statcast.woba_agg(),
    statcast.xwoba_agg(),
)
woba_check.sort("PA", descending=True).head(10)

batter,PA,wOBA,xwOBA
i64,u32,f64,f64
596019,732,0.358528,0.347799
646240,732,0.377368,0.372902
660271,727,0.434632,0.437206
656941,726,0.396727,0.404368
621566,724,0.370943,0.36122
672695,723,0.387941,0.360973
665742,716,0.411286,0.436689
668227,712,0.340793,0.328095
677594,710,0.358393,0.347343


In [9]:
k_rates = statcast.batter_k_rate(pa, min_pa=200)
k_rates.head(10)

batter,PA,K,k_rate
i64,u32,u32,f64
669911,338,132,0.390533
656180,288,110,0.381944
666624,306,109,0.356209
681297,360,128,0.355556
672356,471,162,0.343949
672744,234,80,0.34188
519317,281,96,0.341637
663886,342,116,0.339181
572191,325,110,0.338462


In [10]:
identity_check = disc.select(
    (pl.col("is_swing").sum() == (pl.col("is_whiff").sum() + pl.col("is_contact").sum())).alias("swings_eq_whiffs_plus_contacts")
)
identity_check

swings_eq_whiffs_plus_contacts
bool
true
